# 📊 데이터 분석 입문 - 5편: 종합 프로젝트

## "디지털 도구 활용 방식에 따른 학습 성취도 분석"
### — PISA 2022 데이터를 중심으로

> **이 노트북은 실제 PISA 데이터로 분석하는 전체 과정을 담고 있습니다.**
> 1~4편에서 배운 모든 기술을 총동원합니다.
>
> ⚠️ **PISA 데이터를 먼저 다운받아 Google Drive에 올려두세요.**
> 다운로드: https://www.oecd.org/en/data/datasets/pisa-2022-database.html

---

## 0. 프로젝트 구조

```
1. 연구 배경 & 질문 설정
2. 데이터 불러오기 & 한국 데이터 추출
3. 변수 선택 & 전처리
4. 탐색적 데이터 분석 (EDA)
5. 통계 분석
6. 결과 해석 & 결론
```

---

## 1. 연구 배경 & 질문

### 연구 배경
OECD PISA 2022는 81개국 약 69만 명의 15세 학생을 대상으로 수학, 읽기, 과학 성취도를 평가한 국제 조사입니다.
한국은 186개교 6,931명이 참여했으며, 52개국이 선택적으로 참여한 ICT 설문도 포함되어 있습니다.

### 연구 질문
1. **한국 학생의 ICT(디지털 도구) 활용 현황**은 어떠한가?
2. **학교에서의 ICT 활용 빈도**와 학업 성취도 사이에 관련이 있는가?
3. **ICT 활용 목적(학습용 vs 여가용)**에 따라 성취도 차이가 있는가?
4. ICT를 **능동적으로 활용하는 학생**과 **수동적으로 활용하는 학생**의 성취도는 어떻게 다른가?

In [ ]:
# 라이브러리 설치 & 불러오기
!pip install pyreadstat -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
import pyreadstat
import warnings

# 한글 폰트 설정
!apt-get install -y fonts-nanum > /dev/null 2>&1
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')

print("✅ 준비 완료!")

---
## 2. 데이터 불러오기

### Google Drive 연결

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# ⬇️ 아래 경로를 본인의 파일 위치에 맞게 수정하세요
# PISA 학생 설문 데이터 파일 (.sav)
PISA_PATH = '/content/drive/MyDrive/PISA2022/CY08MSP_STU_QQQ.sav'

print("파일 경로 설정 완료:", PISA_PATH)

In [ ]:
# PISA 데이터 불러오기 (파일이 크므로 시간이 좀 걸립니다)
# 필요한 열만 선택해서 불러오면 훨씬 빠릅니다

필요한_열 = [
    # 기본 정보
    'CNT',          # 국가 코드
    'CNTSCHID',     # 학교 ID  
    'CNTSTUID',     # 학생 ID
    'ST004D01T',    # 성별 (1=여, 2=남)
    
    # 성취도 점수 (Plausible Values)
    'PV1MATH', 'PV2MATH', 'PV3MATH',   # 수학
    'PV1READ', 'PV2READ', 'PV3READ',   # 읽기
    'PV1SCIE', 'PV2SCIE', 'PV3SCIE',   # 과학
    
    # ICT 관련 변수 (학교에서의 디지털 자원 사용)
    'IC170Q01JA',   # 학교 - 데스크톱/노트북
    'IC170Q02JA',   # 학교 - 스마트폰
    'IC170Q03JA',   # 학교 - 태블릿
    'IC170Q04JA',   # 학교 - 인터넷
    'IC170Q05JA',   # 학교 - 교육용 소프트웨어
    'IC170Q06JA',   # 학교 - 학습관리시스템
    'IC170Q07JA',   # 학교 - 시뮬레이션/가상실험
    
    # ICT 관련 변수 (가정에서의 디지털 자원 사용)
    'IC171Q01JA',   # 가정 - 데스크톱/노트북
    'IC171Q02JA',   # 가정 - 스마트폰
    'IC171Q03JA',   # 가정 - 태블릿
    'IC171Q04JA',   # 가정 - 인터넷
    
    # ICT 활용 목적 (학교 활동 관련)
    'IC174Q01JA',   # 데이터 수집/기록
    'IC174Q02JA',   # 프로젝트 계획/관리
    'IC174Q03JA',   # 학습 게임
    'IC174Q04JA',   # 과제 제출
    'IC174Q05JA',   # SNS 브라우징
    
    # 사회경제적 배경
    'ESCS',         # 경제사회문화지위 지수
    
    # 가중치
    'W_FSTUWT',     # 학생 가중치
]

print("불러올 변수:", len(필요한_열), "개")
print("\n데이터를 불러오는 중... (1-3분 소요)")

try:
    df_pisa, meta = pyreadstat.read_sav(PISA_PATH, usecols=필요한_열)
    print(f"✅ 전체 데이터: {df_pisa.shape[0]:,}명, {df_pisa.shape[1]}개 변수")
except FileNotFoundError:
    print("❌ 파일을 찾을 수 없습니다. PISA_PATH를 확인하세요.")
    print("   파일을 아직 다운받지 않았다면:")
    print("   https://www.oecd.org/en/data/datasets/pisa-2022-database.html")

In [ ]:
# 한국 데이터만 추출
df = df_pisa[df_pisa['CNT'] == 'KOR'].copy()
print(f"🇰🇷 한국 학생 수: {len(df):,}명")
print(f"변수 수: {df.shape[1]}개")
df.head()

---
## 3. 변수 전처리

### 3.1 성취도 점수 처리
PISA는 성취도를 **Plausible Values** 여러 개로 제공합니다.
간단한 분석에서는 이들의 **평균**을 사용합니다.

In [ ]:
# 성취도 평균 계산
df['수학'] = df[['PV1MATH', 'PV2MATH', 'PV3MATH']].mean(axis=1)
df['읽기'] = df[['PV1READ', 'PV2READ', 'PV3READ']].mean(axis=1)
df['과학'] = df[['PV1SCIE', 'PV2SCIE', 'PV3SCIE']].mean(axis=1)

print("성취도 기술통계:")
print(df[['수학', '읽기', '과학']].describe().round(1))

In [ ]:
# 성별 변환
df['성별'] = df['ST004D01T'].map({1: '여', 2: '남'})
print("성별 분포:")
print(df['성별'].value_counts())

### 3.2 ICT 변수 이해

ICT 변수의 응답 척도:
- 1 = 전혀 또는 거의 안 함
- 2 = 한 달에 1~2번
- 3 = 일주일에 1~2번
- 4 = 거의 매일
- 5 = 하루에 여러 번
- 6 = 이 자원을 사용할 수 없음 (→ 결측 처리)

In [ ]:
# ICT 변수 확인 & 6번 응답을 결측치로 변환
ict_학교 = ['IC170Q01JA', 'IC170Q02JA', 'IC170Q03JA', 'IC170Q04JA', 
           'IC170Q05JA', 'IC170Q06JA', 'IC170Q07JA']
ict_가정 = ['IC171Q01JA', 'IC171Q02JA', 'IC171Q03JA', 'IC171Q04JA']
ict_활동 = ['IC174Q01JA', 'IC174Q02JA', 'IC174Q03JA', 'IC174Q04JA', 'IC174Q05JA']

# 6 (자원 없음) → NaN
for col in ict_학교 + ict_가정 + ict_활동:
    if col in df.columns:
        df[col] = df[col].replace(6, np.nan)

print("ICT 학교 사용 결측치:")
print(df[ict_학교].isnull().sum())

In [ ]:
# ICT 종합 점수 만들기
# 학교에서의 ICT 사용 종합 (평균)
df['ICT학교종합'] = df[[c for c in ict_학교 if c in df.columns]].mean(axis=1)

# 가정에서의 ICT 사용 종합
df['ICT가정종합'] = df[[c for c in ict_가정 if c in df.columns]].mean(axis=1)

# 전체 ICT 종합
df['ICT전체종합'] = df[['ICT학교종합', 'ICT가정종합']].mean(axis=1)

print("ICT 종합 점수 분포:")
print(df[['ICT학교종합', 'ICT가정종합', 'ICT전체종합']].describe().round(2))

In [ ]:
# ICT 사용 수준 그룹화
df['ICT사용수준'] = pd.cut(df['ICT학교종합'], 
                        bins=[0, 2, 3, 5],
                        labels=['저사용', '중사용', '고사용'])

print("ICT 학교 사용 수준 분포:")
print(df['ICT사용수준'].value_counts())

---
## 4. 탐색적 데이터 분석 (EDA)

### 4.1 성취도 분포

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, (과목, 색) in enumerate([('수학', 'steelblue'), ('읽기', 'coral'), ('과학', 'seagreen')]):
    axes[i].hist(df[과목].dropna(), bins=30, color=색, edgecolor='white', alpha=0.8)
    axes[i].axvline(df[과목].mean(), color='red', linestyle='--', 
                    label=f'평균: {df[과목].mean():.0f}')
    axes[i].set_title(f'{과목} 점수 분포')
    axes[i].set_xlabel('점수')
    axes[i].set_ylabel('학생 수')
    axes[i].legend()

plt.suptitle('🇰🇷 한국 학생 PISA 2022 성취도 분포', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('성취도_분포.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.2 ICT 사용 현황

In [ ]:
# ICT 학교 사용 항목별 평균
ict_항목명 = {
    'IC170Q01JA': '데스크톱/노트북',
    'IC170Q02JA': '스마트폰', 
    'IC170Q03JA': '태블릿',
    'IC170Q04JA': '인터넷',
    'IC170Q05JA': '교육용SW',
    'IC170Q06JA': '학습관리시스템',
    'IC170Q07JA': '시뮬레이션',
}

사용_평균 = {}
for 코드, 이름 in ict_항목명.items():
    if 코드 in df.columns:
        사용_평균[이름] = df[코드].mean()

plt.figure(figsize=(10, 6))
항목 = list(사용_평균.keys())
값 = list(사용_평균.values())
plt.barh(항목, 값, color='steelblue', edgecolor='white')
plt.xlabel('평균 사용 빈도 (1=거의없음 ~ 5=매일여러번)')
plt.title('🇰🇷 한국 학생의 학교 ICT 사용 현황')
plt.xlim(0, 5)
plt.tight_layout()
plt.savefig('ICT사용현황.png', dpi=300, bbox_inches='tight')
plt.show()

### 4.3 ICT 사용 수준별 성취도 비교

In [ ]:
# 그룹별 평균
그룹별 = df.groupby('ICT사용수준')[['수학', '읽기', '과학']].mean().round(1)
print("ICT 사용 수준별 평균 성취도:")
print(그룹별)

# 시각화
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(그룹별))
width = 0.25

ax.bar(x - width, 그룹별['수학'], width, label='수학', color='steelblue')
ax.bar(x, 그룹별['읽기'], width, label='읽기', color='coral')
ax.bar(x + width, 그룹별['과학'], width, label='과학', color='seagreen')

ax.set_xlabel('ICT 사용 수준')
ax.set_ylabel('평균 점수')
ax.set_title('ICT 학교 사용 수준별 평균 성취도')
ax.set_xticks(x)
ax.set_xticklabels(그룹별.index)
ax.legend()

plt.tight_layout()
plt.savefig('ICT수준별_성취도.png', dpi=300, bbox_inches='tight')
plt.show()

---
## 5. 통계 분석

### 5.1 상관분석

In [ ]:
# ICT 종합 사용과 성취도의 상관
print("=== 상관분석 결과 ===\n")

for 과목 in ['수학', '읽기', '과학']:
    유효 = df[['ICT학교종합', 과목]].dropna()
    r, p = stats.pearsonr(유효['ICT학교종합'], 유효[과목])
    유의 = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    print(f"ICT 학교 사용 vs {과목}: r = {r:.3f}, p = {p:.4f} {유의}")

In [ ]:
# 산점도 + 추세선
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, (과목, 색) in enumerate([('수학', 'steelblue'), ('읽기', 'coral'), ('과학', 'seagreen')]):
    유효 = df[['ICT학교종합', 과목]].dropna()
    r, p = stats.pearsonr(유효['ICT학교종합'], 유효[과목])
    
    axes[i].scatter(유효['ICT학교종합'], 유효[과목], alpha=0.3, color=색, s=10)
    
    # 추세선
    z = np.polyfit(유효['ICT학교종합'], 유효[과목], 1)
    p_line = np.poly1d(z)
    x_line = np.linspace(1, 5, 100)
    axes[i].plot(x_line, p_line(x_line), 'r--', linewidth=2)
    
    axes[i].set_xlabel('ICT 학교 사용 (종합)')
    axes[i].set_ylabel(f'{과목} 점수')
    axes[i].set_title(f'{과목} (r={r:.3f}, p={p:.4f})')

plt.suptitle('ICT 학교 사용과 성취도의 관계', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('상관분석_산점도.png', dpi=300, bbox_inches='tight')
plt.show()

### 5.2 그룹 비교 (ANOVA)

In [ ]:
# ICT 사용 수준 3그룹의 수학 점수 ANOVA
저 = df[df['ICT사용수준'] == '저사용']['수학'].dropna()
중 = df[df['ICT사용수준'] == '중사용']['수학'].dropna()
고 = df[df['ICT사용수준'] == '고사용']['수학'].dropna()

F, p = stats.f_oneway(저, 중, 고)
print("=== 일원분산분석 (ANOVA) ===")
print(f"\nICT 사용 수준별 수학 점수:")
print(f"  저사용 (n={len(저)}): M={저.mean():.1f}, SD={저.std():.1f}")
print(f"  중사용 (n={len(중)}): M={중.mean():.1f}, SD={중.std():.1f}")
print(f"  고사용 (n={len(고)}): M={고.mean():.1f}, SD={고.std():.1f}")
print(f"\nF({2}, {len(저)+len(중)+len(고)-3}) = {F:.2f}, p = {p:.4f}")
if p < 0.05:
    print("✅ 세 그룹 간 유의한 차이가 있습니다.")

### 5.3 성별 × ICT 교차 분석

In [ ]:
# 성별에 따른 ICT 사용 수준 분포
교차표 = pd.crosstab(df['성별'], df['ICT사용수준'], normalize='index') * 100
print("성별별 ICT 사용 수준 비율 (%):")
print(교차표.round(1))

# 카이제곱 검정
교차표_raw = pd.crosstab(df['성별'], df['ICT사용수준'])
chi2, p, dof, exp = stats.chi2_contingency(교차표_raw)
print(f"\nχ² = {chi2:.2f}, p = {p:.4f}")

---
## 6. 결과 해석 & 보고서용 요약

### ✏️ 여기가 가장 중요합니다!
아래 템플릿을 채워서 보고서의 결론 부분을 작성해보세요.

In [ ]:
# 결과 종합 출력
print("=" * 70)
print("📋 연구 결과 종합 요약")
print("=" * 70)

print(f"""
■ 연구 제목: 디지털 도구 활용 방식에 따른 학습 성취도 분석
■ 데이터: PISA 2022 한국 학생 ({len(df):,}명)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. 한국 학생의 ICT 활용 현황
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   - 학교에서의 ICT 사용 종합: 평균 {df['ICT학교종합'].mean():.2f} / 5.0

2. ICT 사용과 성취도의 관계
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   [아래 상관분석 결과를 보고 직접 해석을 작성하세요]

3. 시사점
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   [데이터에서 발견한 패턴을 바탕으로 시사점을 작성하세요]
   
   예시:
   - 단순히 ICT를 많이 사용하는 것이 높은 성취로 이어지는가?
   - 어떤 유형의 ICT 활용이 성취도와 더 관련이 있는가?
   - 이 결과를 AI 활용 맥락에서 어떻게 해석할 수 있는가?
""")

---
## 🎯 다음 단계: 자체 설문으로 확장

이 PISA 분석에서 발견한 패턴을 **가설**로 삼고,
**AI 활용**이라는 더 구체적인 맥락에서 자체 설문으로 검증합니다.

### 가설 예시:
- PISA에서 "능동적 ICT 활용이 성취도와 양의 상관" → 
  "AI를 탐구적으로 사용하는 학생이 학습 만족도가 높을 것이다"
- PISA에서 "과도한 ICT 사용은 오히려 부정적" →
  "AI 의존도가 높은 학생은 자기주도 학습력이 낮을 것이다"

### 설문에 포함할 문항 (예시):
1. AI 도구(ChatGPT 등)를 얼마나 자주 사용합니까? (1-5)
2. AI를 주로 어떤 목적으로 사용합니까? (답 확인 / 개념 이해 / 아이디어 발상 / ...)
3. AI 사용 후 해당 내용을 스스로 다시 풀어보십니까? (1-5)
4. 자신의 학업 성취도를 어떻게 평가합니까? (1-5)
5. AI 사용 전후로 학습 방식에 변화가 있었습니까?

---

## 📝 전체 시리즈 완료!

축하합니다! 🎉 이제 당신은:
- Python으로 데이터를 불러오고 탐색할 수 있고
- 그래프로 패턴을 시각화할 수 있고
- 결측치와 이상치를 처리할 수 있고
- 상관분석, t-검정, ANOVA로 가설을 검증할 수 있고
- 분석 결과를 보고서 형식으로 정리할 수 있습니다.

이 기술은 PISA 프로젝트뿐 아니라, **어떤 데이터 분석 프로젝트**에도 적용할 수 있습니다!